<h1><b>Extreme Gradient Boosting</b><br>
<i>Conjunto de dados Diabetes Health Indicators</i></h1>

<b>Objetivo do modelo:</b><br>
Prever a probabilidade de um paciente ter diabetes sem o uso de variáveis clinicas

### Importação das Bibliotecas

In [41]:
import numpy as np
import pandas as pd
import shap

from xgboost import XGBClassifier

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeClassifier, export_text

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

### Configurações do experimento

In [42]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET = "diagnosed_diabetes"
DATA_PATH = "./../../data/raw/diabetes_health_indicators.csv"

### Carregamento dos dados

In [43]:
df_columns = [
    "age",
    "gender",
    "ethnicity",
    "education_level",
    "income_level",
    "employment_status",
    "smoking_status",
    "alcohol_consumption_per_week",
    "physical_activity_minutes_per_week",
    "diet_score",
    "sleep_hours_per_day",
    "screen_time_hours_per_day",
    "bmi",
    TARGET
]

df = pd.read_csv(DATA_PATH, usecols=df_columns)

print(f"{df.shape[0]} registros e {df.shape[1]} variáveis")

100000 registros e 14 variáveis


### Discretização exploratória com árvore

In [44]:
tree_discretizer = DecisionTreeClassifier(
    max_leaf_nodes=4,
    random_state=RANDOM_STATE
)

tree_discretizer.fit(df[["age"]], df[TARGET])

thresholds = sorted(
    [t for t in tree_discretizer.tree_.threshold if t != -2]
)

thresholds = [round(t) for t in thresholds]

print("Pontos de corte sugeridos:", thresholds)
print(export_text(tree_discretizer, feature_names=["age"]))

Pontos de corte sugeridos: [28, 48, 68]
|--- age <= 48.50
|   |--- age <= 28.50
|   |   |--- class: 0
|   |--- age >  28.50
|   |   |--- class: 1
|--- age >  48.50
|   |--- age <= 67.50
|   |   |--- class: 1
|   |--- age >  67.50
|   |   |--- class: 1



Criando as faixas

In [45]:
bins = [-np.inf] + thresholds + [np.inf]
labels = [f"Faixa_{i}" for i in range(len(bins) - 1)]

df["age_binned"] = pd.cut(
    df["age"],
    bins=bins,
    labels=labels
)

Estatística por faixa

In [46]:
age_stats = (
    df.groupby("age_binned")[TARGET]
    .agg(["count", "mean"])
    .rename(columns={
        "count": "n",
        "mean": "diabetes_rate"
    })
)

print(age_stats)

                n  diabetes_rate
age_binned                      
Faixa_0      8932           0.47
Faixa_1     37278           0.56
Faixa_2     41310           0.63
Faixa_3     12480           0.72


### Separação treino/teste

In [47]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

### Definição das features

In [48]:
numeric_features = [
    "age",
    "alcohol_consumption_per_week",
    "physical_activity_minutes_per_week",
    "diet_score",
    "sleep_hours_per_day",
    "screen_time_hours_per_day",
    "bmi"
]

categorical_features = [
    "age_binned",
    "gender",
    "ethnicity",
    "education_level",
    "income_level",
    "employment_status",
    "smoking_status"
]

### ColumnTransformer

In [49]:
preprocessor = ColumnTransformer([
    ("cat",
     OneHotEncoder(
         drop="first",
         handle_unknown="ignore"
     ),
     categorical_features)
], remainder="passthrough")

### Pipeline do modelo

In [50]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

### Espaço de hiperparâmetros

In [51]:
param_dist = {
    "classifier__n_estimators": [200, 400, 600, 800],
    "classifier__max_depth": [3, 4, 5, 6],
    "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "classifier__gamma": [0, 0.1, 0.3],
    "classifier__min_child_weight": [1, 3, 5]
}

### Cross Validation

In [52]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

### Random Search

In [53]:
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__colsample_bytree': [0.7, 0.8, ...], 'classifier__gamma': [0, 0.1, ...], 'classifier__learning_rate': [0.01, 0.05, ...], 'classifier__max_depth': [3, 4, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscro

### Melhor modelo

In [54]:
best_model = random_search.best_estimator_

print("Melhores parâmetros:")
print(random_search.best_params_)

print("ROC AUC médio (CV):")
print(random_search.best_score_)

Melhores parâmetros:
{'classifier__subsample': 0.8, 'classifier__n_estimators': 400, 'classifier__min_child_weight': 5, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.01, 'classifier__gamma': 0, 'classifier__colsample_bytree': 1.0}
ROC AUC médio (CV):
0.6106305525506939


### Avaliação no conjunto de teste

In [55]:
y_pred_prob = best_model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_pred_prob)

print("ROC AUC Teste:", roc_auc)

ROC AUC Teste: 0.6012474166666667


### Classification Report

In [56]:
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.56      0.17      0.26      8000
           1       0.62      0.91      0.74     12000

    accuracy                           0.61     20000
   macro avg       0.59      0.54      0.50     20000
weighted avg       0.60      0.61      0.55     20000



### Confusion Matrix

In [57]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[ 1336  6664]
 [ 1051 10949]]


### Avaliação em múltiplos thresholds

In [62]:
thresholds = [0.3, 0.4, 0.5, 0.6]

results = []

for t in thresholds:

    y_pred = (y_pred_prob >= t).astype(int)

    results.append({
        "threshold": t,
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    })

threshold_analysis = pd.DataFrame(results)

print(threshold_analysis)

   threshold  precision  recall   f1
0       0.30       0.60    1.00 0.75
1       0.40       0.60    0.99 0.75
2       0.50       0.62    0.91 0.74
3       0.60       0.67    0.57 0.62


### Estabilidade do Cross Validation

In [59]:
cv_results = pd.DataFrame(random_search.cv_results_)

cv_results = cv_results[
    [
        "mean_test_score",
        "std_test_score",
        "param_classifier__n_estimators",
        "param_classifier__max_depth",
        "param_classifier__learning_rate"
    ]
]

cv_results.sort_values(
    ["mean_test_score", "std_test_score"],
    ascending=[False, True]
).head(10)

,mean_test_score,std_test_score,param_classifier__n_estimators,param_classifier__max_depth,param_classifier__learning_rate
12,0.61,0.00,400,3,0.01
2,0.61,0.00,800,3,0.01
13,0.61,0.00,200,5,0.01
1,0.61,0.00,800,4,0.01
11,0.61,0.00,200,3,0.01
10,0.61,0.00,400,6,0.01
5,0.61,0.00,600,5,0.01
17,0.61,0.00,200,3,0.10
23,0.61,0.00,600,6,0.01
27,0.61,0.00,800,6,0.01


### Interpretabilidade com SHAP

In [60]:
X_train_transformed = best_model.named_steps[
    "preprocessor"
].transform(X_train)

feature_names = best_model.named_steps[
    "preprocessor"
].get_feature_names_out()

explainer = shap.TreeExplainer(
    best_model.named_steps["classifier"]
)

shap_values = explainer.shap_values(
    X_train_transformed
)

shap_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0)
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

print(shap_importance.head(10))

                                         feature  mean_abs_shap
0                                 remainder__age           0.21
1  remainder__physical_activity_minutes_per_week           0.14
2                                 remainder__bmi           0.12
3                          remainder__diet_score           0.04
4           remainder__screen_time_hours_per_day           0.02
5                        cat__age_binned_Faixa_3           0.01
6                        cat__age_binned_Faixa_2           0.01
7                        cat__age_binned_Faixa_1           0.00
8                 remainder__sleep_hours_per_day           0.00
9                 cat__employment_status_Retired           0.00
